# 09 · Учитель-промпт и несколько слоёв: цель без учителя-LoRA, которая переносится

**Зачем** (JOURNAL, 2026-10-31, прогон 08). Аффинная поправка в точке вызова с LoRA-учителем переносится
на новые типы требований (0.78 против 0.85 у LoRA), а без учителя — нет (0.61): узкое место — цель.
Здесь две цели без градиентного обучения и без LoRA:

| метод | цель | слои |
|---|---|---|
| `linmap_prompt` | Δh = h(та же модель с промптом explicit_state) − h(база), где учитель решил верно | один, по сетке |
| `linmap_stack` | цель `linmap` (центроид верных состояний базы) | все слои сразу |
| `linmap_prompt_stack` | цель `linmap_prompt` | все слои сразу |

Слои сразу — идея SimplexLoRA (2610.37206): бюджет ранга R = 8 × число слоёв и сила A распределены по слоям
на симплексе — **поровну** или **по важности** слоя (прирост лучшей одно-слойной конфигурации на val).

Данные — те же 110 промптов train; выбор на val; сетка силы расширена (1.5 была на краю). Проверка —
на **свежем** наборе переноса (seed 303): набор 202 уже повлиял на выбор этих методов. Результат **разведочный**.

In [1]:
%load_ext autoreload
%autoreload 2
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # Mac: неподдержанные операции -> CPU

import pandas as pd
from IPython.display import Markdown, display
pd.set_option("display.precision", 3)
pd.set_option("display.width", 200)

from steering_tool_feedback.config import ExperimentConfig
from steering_tool_feedback.pipeline import Experiment
from steering_tool_feedback import journal, viz

In [2]:
cfg = ExperimentConfig.make(
    name="prompt-teacher-qwen3-1.7b", hypothesis="explore", notebook="notebooks/09_prompt_teacher.ipynb",
    notes="Учитель-промпт (explicit_state) и несколько слоёв с бюджетом по важности (SimplexLoRA) против LoRA.",
    overrides={
        "model.name": "Qwen/Qwen3-1.7B",
        "data.n_tasks": 160,
        "data.generator": "v3",
        "eval.methods": ["baseline", "steering", "lora", "explicit_state", "linmap_lora", "linmap",
                         "linmap_prompt", "linmap_stack", "linmap_prompt_stack"],
        "steer.alphas": [0.25, 0.5, 1.0, 2.0, 4.0],
        "act.ranks": [8, 16],
        "act.alphas": [1.0, 1.5, 2.0, 3.0],
        "episode.on_val": False,
        "transfer.seed": 303,
        "transfer.episode_methods": ["baseline", "lora", "linmap", "linmap_prompt", "linmap_stack",
                                     "linmap_prompt_stack"],
    },
)
exp = Experiment(cfg)
exp.run("fit")
exp.results_table("val")

/Users/admin/Desktop/steering-tool-feedback/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[01:02:05] run 20261031-010203_prompt-teacher-qwen3-1.7b_40692b  ->  results/runs/20261031-010203_prompt-teacher-qwen3-1.7b_40692b
[01:02:05] data: {'train': 79, 'val': 31, 'test': 50}  types={'invariant': 41, 'changing': 112, 'stress': 7}  positions={'early': 114, 'late': 46}


baseline/train:   0%|          | 0/158 [00:00<?, ?it/s]

[01:02:05] loading model Qwen/Qwen3-1.7B …


W1031 01:02:09.340000 7953 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

... (7,374 outputs left out)

linmap_stack/val:   0%|          | 0/62 [00:00<?, ?it/s]

linmap_prompt_stack/val:   0%|          | 0/62 [00:00<?, ?it/s]

[01:52:41] val: baseline 0.621; steering 0.655; lora 0.948; explicit_state 0.552; linmap_lora 0.845; linmap 0.862; linmap_prompt 0.741; linmap_stack 0.793; linmap_prompt_stack 0.707


,method,success,pass^1,sensitivity,specificity,diff_vs_base,ci,fixed,broken,episode,ep_diff,ep_ci,stress,tokens,calls,p50_ms
0,baseline,0.621,0.621,0.526,0.4,NaN,,NaN,NaN,None,None,,0.50,831,10.0,400.6
1,steering,0.655,0.655,0.632,0.4,0.034,"[+0.000, +0.086]",2.0,0.0,None,None,,0.50,831,10.0,410.2
2,lora,0.948,0.948,1.000,0.8,0.328,"[+0.190, +0.466]",19.0,0.0,None,None,,0.00,831,10.0,299.0
3,explicit_state,0.552,0.552,0.474,0.1,-0.069,"[-0.207, +0.069]",6.0,10.0,None,None,,0.25,1932,11.0,3443.7
4,linmap_lora,0.845,0.845,0.789,0.7,0.224,"[+0.086, +0.379]",18.0,5.0,None,None,,0.50,831,10.0,425.8
5,linmap,0.862,0.862,0.842,0.6,0.241,"[+0.121, +0.379]",16.0,2.0,None,None,,0.50,831,10.0,410.6
6,linmap_prompt,0.741,0.741,0.632,0.6,0.121,"[+0.017, +0.241]",9.0,2.0,None,None,,0.50,831,10.0,297.4
7,linmap_stack,0.793,0.793,0.842,0.5,0.172,"[+0.069, +0.293]",10.0,0.0,None,None,,0.50,831,10.0,307.4
8,linmap_prompt_stack,0.707,0.707,0.632,0.5,0.086,"[+0.000, +0.190]",6.0,1.0,None,None,,0.50,831,10.0,301.4


### Подбор на val

In [3]:
f = exp.state["fitted"]
t = f.get("linmap_prompt", {})
print(f"учитель {t.get('teacher')} на train: успех {t.get('teacher_train_success', float('nan')):.3f} "
      f"(база {t.get('base_train_success', float('nan')):.3f})")
for m in ("linmap_prompt", "linmap", "linmap_lora"):
    t = exp.fit_table(m)
    if t.empty:
        print(m, f[m].get("reason")); continue
    display(Markdown(f"**{m}** — лучшие конфигурации на val"))
    cols = ["layer", "rank", "alpha", "train_r2", "success", "sensitivity", "specificity", "spec_drop", "fixed", "broken"]
    display(t.sort_values("success", ascending=False)[cols].head(8).round(3))

учитель explicit_state на train: успех 0.736 (база 0.755)


**linmap_prompt** — лучшие конфигурации на val

,layer,rank,alpha,train_r2,success,sensitivity,specificity,spec_drop,fixed,broken
27,22,8,3.0,0.630,0.741,0.684,0.5,-0.1,8,1
19,20,8,3.0,0.668,0.741,0.632,0.6,-0.2,9,2
43,26,8,3.0,0.611,0.741,0.684,0.5,-0.1,9,2
35,24,8,3.0,0.624,0.741,0.684,0.5,-0.1,9,2
37,24,16,1.5,0.680,0.724,0.684,0.5,-0.1,7,1
30,22,16,2.0,0.686,0.724,0.684,0.5,-0.1,7,1
31,22,16,3.0,0.686,0.724,0.684,0.5,-0.1,7,1
33,24,8,1.5,0.624,0.724,0.684,0.5,-0.1,7,1


**linmap** — лучшие конфигурации на val

,layer,rank,alpha,train_r2,success,sensitivity,specificity,spec_drop,fixed,broken
9,18,8,1.5,0.547,0.862,0.842,0.6,-0.2,16,2
13,18,16,1.5,0.568,0.862,0.789,0.8,-0.4,17,3
21,20,16,1.5,0.561,0.828,0.737,0.6,-0.2,15,3
14,18,16,2.0,0.568,0.828,0.684,0.7,-0.3,18,6
38,24,16,2.0,0.567,0.828,0.737,0.6,-0.2,15,3
35,24,8,3.0,0.553,0.828,0.684,0.9,-0.5,18,6
12,18,16,1.0,0.568,0.810,0.842,0.5,-0.1,11,0
1,16,8,1.5,0.516,0.810,0.789,0.6,-0.2,12,1


**linmap_lora** — лучшие конфигурации на val

,layer,rank,alpha,train_r2,success,sensitivity,specificity,spec_drop,fixed,broken
45,26,16,1.5,0.963,0.845,0.789,0.7,-0.3,18,5
37,24,16,1.5,0.944,0.828,0.632,0.8,-0.4,16,4
14,18,16,2.0,0.891,0.810,0.789,0.6,-0.2,11,0
7,16,16,3.0,0.911,0.810,0.789,0.6,-0.2,11,0
11,18,8,3.0,0.839,0.810,0.737,0.6,-0.2,12,1
13,18,16,1.5,0.891,0.793,0.789,0.5,-0.1,10,0
9,18,8,1.5,0.839,0.793,0.789,0.5,-0.1,10,0
10,18,8,2.0,0.839,0.793,0.789,0.5,-0.1,10,0


In [4]:
for m in ("linmap_stack", "linmap_prompt_stack"):
    t = exp.fit_table(m)
    if t.empty:
        print(m, f[m].get("reason")); continue
    imp = {L: round(g, 3) for L, g in f[m]["importance"].items()}
    display(Markdown(f"**{m}** — равномерно / по важности; важность слоя (прирост на val): {imp}"))
    cols = ["alloc", "alpha", "layers", "ranks", "weights", "success", "sensitivity", "specificity",
            "spec_drop", "fixed", "broken"]
    display(t[cols].round(3))

**linmap_stack** — равномерно / по важности; важность слоя (прирост на val): {16: 0.19, 18: 0.241, 20: 0.207, 22: 0.138, 24: 0.207, 26: 0.172}

,alloc,alpha,layers,ranks,weights,success,sensitivity,specificity,spec_drop,fixed,broken
0,uniform,1.0,"[16, 18, 20, 22, 24, 26]","[8, 8, 8, 8, 8, 8]","[0.167, 0.167, 0.167, 0.167, 0.167, 0.167]",0.776,0.789,0.5,-0.1,9,0
1,uniform,1.5,"[16, 18, 20, 22, 24, 26]","[8, 8, 8, 8, 8, 8]","[0.167, 0.167, 0.167, 0.167, 0.167, 0.167]",0.793,0.842,0.5,-0.1,10,0
2,uniform,2.0,"[16, 18, 20, 22, 24, 26]","[8, 8, 8, 8, 8, 8]","[0.167, 0.167, 0.167, 0.167, 0.167, 0.167]",0.776,0.789,0.5,-0.1,10,1
3,uniform,3.0,"[16, 18, 20, 22, 24, 26]","[8, 8, 8, 8, 8, 8]","[0.167, 0.167, 0.167, 0.167, 0.167, 0.167]",0.793,0.737,0.6,-0.2,15,5
4,importance,1.0,"[16, 18, 20, 22, 24, 26]","[8, 10, 9, 6, 9, 7]","[0.164, 0.209, 0.179, 0.119, 0.179, 0.149]",0.776,0.789,0.5,-0.1,9,0
5,importance,1.5,"[16, 18, 20, 22, 24, 26]","[8, 10, 9, 6, 9, 7]","[0.164, 0.209, 0.179, 0.119, 0.179, 0.149]",0.793,0.842,0.5,-0.1,10,0
6,importance,2.0,"[16, 18, 20, 22, 24, 26]","[8, 10, 9, 6, 9, 7]","[0.164, 0.209, 0.179, 0.119, 0.179, 0.149]",0.793,0.842,0.5,-0.1,11,1
7,importance,3.0,"[16, 18, 20, 22, 24, 26]","[8, 10, 9, 6, 9, 7]","[0.164, 0.209, 0.179, 0.119, 0.179, 0.149]",0.793,0.684,0.6,-0.2,15,5


**linmap_prompt_stack** — равномерно / по важности; важность слоя (прирост на val): {16: 0.086, 18: 0.103, 20: 0.121, 22: 0.121, 24: 0.121, 26: 0.121}

,alloc,alpha,layers,ranks,weights,success,sensitivity,specificity,spec_drop,fixed,broken
0,uniform,1.0,"[16, 18, 20, 22, 24, 26]","[8, 8, 8, 8, 8, 8]","[0.167, 0.167, 0.167, 0.167, 0.167, 0.167]",0.707,0.632,0.5,-0.1,6,1
1,uniform,1.5,"[16, 18, 20, 22, 24, 26]","[8, 8, 8, 8, 8, 8]","[0.167, 0.167, 0.167, 0.167, 0.167, 0.167]",0.707,0.632,0.5,-0.1,6,1
2,uniform,2.0,"[16, 18, 20, 22, 24, 26]","[8, 8, 8, 8, 8, 8]","[0.167, 0.167, 0.167, 0.167, 0.167, 0.167]",0.707,0.632,0.5,-0.1,6,1
3,uniform,3.0,"[16, 18, 20, 22, 24, 26]","[8, 8, 8, 8, 8, 8]","[0.167, 0.167, 0.167, 0.167, 0.167, 0.167]",0.707,0.632,0.5,-0.1,6,1
4,importance,1.0,"[16, 18, 20, 22, 24, 26]","[6, 7, 9, 9, 9, 9]","[0.128, 0.154, 0.179, 0.179, 0.179, 0.179]",0.707,0.632,0.5,-0.1,6,1
5,importance,1.5,"[16, 18, 20, 22, 24, 26]","[6, 7, 9, 9, 9, 9]","[0.128, 0.154, 0.179, 0.179, 0.179, 0.179]",0.707,0.632,0.5,-0.1,6,1
6,importance,2.0,"[16, 18, 20, 22, 24, 26]","[6, 7, 9, 9, 9, 9]","[0.128, 0.154, 0.179, 0.179, 0.179, 0.179]",0.707,0.632,0.5,-0.1,6,1
7,importance,3.0,"[16, 18, 20, 22, 24, 26]","[6, 7, 9, 9, 9, 9]","[0.128, 0.154, 0.179, 0.179, 0.179, 0.179]",0.707,0.632,0.5,-0.1,6,1


### Свежий набор переноса (seed 303): знакомые и новые типы требований

In [5]:
exp.transfer()
for g in ("seen", "unseen"):
    display(Markdown(f"**{g}**"))
    display(exp.results_table(f"transfer:{g}"))

baseline/transfer:seen:   0%|          | 0/100 [00:00<?, ?it/s]

baseline/transfer:seen:   1%|          | 1/100 [00:00<00:48,  2.05it/s]

baseline/transfer:seen:   2%|▏         | 2/100 [00:00<00:47,  2.06it/s]

baseline/transfer:seen:   3%|▎         | 3/100 [00:01<00:46,  2.09it/s]

baseline/transfer:seen:   4%|▍         | 4/100 [00:01<00:45,  2.10it/s]

baseline/transfer:seen:   5%|▌         | 5/100 [00:02<00:39,  2.42it/s]

... (3,027 outputs left out)

**seen**

,method,success,pass^1,sensitivity,specificity,diff_vs_base,ci,fixed,broken,episode,ep_diff,ep_ci,stress,tokens,calls,p50_ms
0,baseline,0.56,0.56,0.132,0.583,NaN,,NaN,NaN,0.56,NaN,,None,861,10.0,299.9
1,steering,0.55,0.55,0.132,0.583,-0.01,"[-0.060, +0.040]",5.0,6.0,NaN,NaN,,None,861,10.0,302.6
2,lora,0.85,0.85,0.632,1.000,0.29,"[+0.190, +0.390]",33.0,4.0,0.76,0.20,"[+0.100, +0.300]",None,861,10.0,309.2
3,explicit_state,0.72,0.72,0.368,0.833,0.16,"[+0.070, +0.250]",19.0,3.0,NaN,NaN,,None,2032,11.0,4371.8
4,linmap_lora,0.68,0.68,0.316,0.667,0.12,"[+0.030, +0.220]",26.0,14.0,NaN,NaN,,None,861,10.0,301.2
5,linmap,0.76,0.76,0.605,0.667,0.20,"[+0.120, +0.280]",28.0,8.0,0.68,0.12,"[+0.040, +0.200]",None,861,10.0,309.0
6,linmap_prompt,0.70,0.70,0.289,0.833,0.14,"[+0.050, +0.230]",19.0,5.0,0.63,0.07,"[-0.010, +0.160]",None,861,10.0,311.8
7,linmap_stack,0.69,0.69,0.342,0.750,0.13,"[+0.060, +0.210]",15.0,2.0,0.66,0.10,"[+0.030, +0.180]",None,861,10.0,323.9
8,linmap_prompt_stack,0.66,0.66,0.289,0.750,0.10,"[+0.030, +0.180]",11.0,1.0,0.64,0.08,"[+0.020, +0.160]",None,861,10.0,323.0


**unseen**

,method,success,pass^1,sensitivity,specificity,diff_vs_base,ci,fixed,broken,episode,ep_diff,ep_ci,stress,tokens,calls,p50_ms
0,baseline,0.45,0.45,0.054,0.385,NaN,,NaN,NaN,0.33,NaN,,None,840,10.0,297.5
1,steering,0.50,0.50,0.108,0.462,0.05,"[-0.030, +0.130]",9.0,4.0,NaN,NaN,,None,840,10.0,303.1
2,lora,0.88,0.88,0.757,0.846,0.43,"[+0.320, +0.540]",45.0,2.0,0.85,0.52,"[+0.390, +0.640]",None,840,10.0,312.6
3,explicit_state,0.91,0.91,0.838,0.846,0.46,"[+0.360, +0.560]",50.0,4.0,NaN,NaN,,None,1988,11.0,4141.2
4,linmap_lora,0.77,0.77,0.568,0.692,0.32,"[+0.180, +0.460]",40.0,8.0,NaN,NaN,,None,840,10.0,298.8
5,linmap,0.64,0.64,0.243,0.692,0.19,"[+0.100, +0.280]",20.0,1.0,0.46,0.13,"[+0.010, +0.250]",None,840,10.0,300.9
6,linmap_prompt,0.83,0.83,0.622,0.846,0.38,"[+0.270, +0.500]",41.0,3.0,0.72,0.39,"[+0.260, +0.520]",None,840,10.0,300.8
7,linmap_stack,0.61,0.61,0.162,0.692,0.16,"[+0.080, +0.250]",17.0,1.0,0.41,0.08,"[-0.020, +0.180]",None,840,10.0,311.9
8,linmap_prompt_stack,0.64,0.64,0.270,0.692,0.19,"[+0.110, +0.280]",19.0,0.0,0.57,0.24,"[+0.140, +0.340]",None,840,10.0,312.2


### Цена: подготовка и вывод

In [6]:
d = exp.backend.hidden_size
cost = []
lr = f["lora"]["row"]
cost.append({"method": "lora", "prep": "градиент по весам (копия fp32)", "prep_seconds": lr["train_seconds"],
             "params": lr["n_trainable"]})
for m in ("linmap_lora", "linmap", "linmap_prompt"):
    r, c = f[m]["row"], f[m].get("cost", {})
    if r:
        how = {"linmap_lora": "закрытая форма + учитель LoRA (+ цена LoRA)", "linmap": "закрытая форма",
               "linmap_prompt": "закрытая форма + учитель-промпт (генерация)"}[m]
        cost.append({"method": m, "prep": how, "prep_seconds": c.get("acts_seconds", 0) + c.get("fit_seconds", 0),
                     "params": 2 * d * r["rank"] + d})
for m, parent in (("linmap_stack", "linmap"), ("linmap_prompt_stack", "linmap_prompt")):
    r = f[m]["row"]
    if r:
        c = f[parent].get("cost", {})
        cost.append({"method": m, "prep": f"как {parent}, {len(r['layers'])} слоёв ({r['alloc']})",
                     "prep_seconds": c.get("acts_seconds", 0) + c.get("fit_seconds", 0),
                     "params": sum(2 * d * k + d for k in r["ranks"])})
cost = pd.DataFrame(cost)
inf = exp.results_table("transfer:seen").set_index("method")[["tokens", "calls", "p50_ms"]]
cost.merge(inf, left_on="method", right_index=True, how="outer").reset_index(drop=True)

,method,prep,prep_seconds,params,tokens,calls,p50_ms
0,baseline,NaN,NaN,NaN,861,10.0,299.9
1,explicit_state,NaN,NaN,NaN,2032,11.0,4371.8
2,linmap,закрытая форма,108.1,3.482e+04,861,10.0,309.0
3,linmap_lora,закрытая форма + учитель LoRA (+ цена LoRA),223.9,6.758e+04,861,10.0,301.2
4,linmap_prompt,закрытая форма + учитель-промпт (генерация),655.3,3.482e+04,861,10.0,311.8
5,linmap_prompt_stack,"как linmap_prompt, 6 слоёв (uniform)",655.3,2.089e+05,861,10.0,323.0
6,linmap_stack,"как linmap, 6 слоёв (uniform)",108.1,2.089e+05,861,10.0,323.9
7,lora,градиент по весам (копия fp32),671.9,3.211e+06,861,10.0,309.2
8,steering,NaN,NaN,NaN,861,10.0,302.6


## Вывод → журнал

In [7]:
display(Markdown(exp.report(
    conclusion="""РАЗВЕДКА (hypothesis=explore, test не открыт). Цели для аффинной поправки в точке вызова без LoRA-учителя, Qwen3-1.7B, v3, те же 110 промптов train; перенос — на свежем наборе seed 303 (набор 202 повлиял на выбор методов).

1) Учитель-промпт (linmap_prompt, выбран L20, ранг 8, α=3.0 — край сетки). Сам учитель explicit_state на train не лучше базы (0.736 против 0.755), на val хуже (0.552, специф. 0.1) — поэтому на val linmap_prompt 0.741 уступает linmap 0.862. Но на переносе (успех / чувств. / специф.; эпизоды):
- новые типы требований: база 0.45/0.05/0.38 (эп. 0.33); LoRA 0.88/0.76/0.85 (0.85); explicit_state 0.91/0.84/0.85; linmap_prompt 0.83/0.62/0.85 (0.72, +0.39 [+0.26, +0.52]); linmap_lora 0.77/0.57/0.69; linmap 0.64/0.24/0.69 (0.46).
- знакомые типы: база 0.56/0.13/0.58 (0.56); LoRA 0.85/0.63/1.00 (0.76); explicit_state 0.72/0.37/0.83; linmap 0.76/0.61/0.67 (0.68); linmap_prompt 0.70/0.29/0.83 (0.63, +0.07, интервал включает 0); linmap_lora 0.68/0.32/0.67.
На новых типах поправка от учителя-промпта даёт ~90% одношагового прироста LoRA (+0.38 против +0.43) и ~75% в эпизодах (+0.39 против +0.52) без градиента и без LoRA; при выводе цена как у базы (~300 мс, 861 токен), а сам explicit_state — 4.1 с и 2.4× токенов. Цели дополняют друг друга: центроид верных состояний (linmap) лучше на знакомых типах, учитель-промпт — на новых.

2) Val не предсказал перенос: на val только знакомые типы, выбор по нему не видит переносимости (linmap_prompt на val ниже linmap, на новых типах выше на 0.19).

3) Повтор на свежем наборе: linmap 0.76 / 0.64 (в 08: 0.85 / 0.61), linmap_lora 0.68 / 0.77 (0.73 / 0.78) — картина та же.

4) Несколько слоёв (идея SimplexLoRA): важность слоёв почти ровная (прирост одного слоя на val 0.14–0.24 для linmap, 0.09–0.12 для linmap_prompt), доли 0.12–0.21 против равных 0.167 — оба распределения дают одно и то же, выбрано равномерное. Стек хуже лучшего одного слоя и на val, и на переносе (новые типы: linmap_stack 0.61, linmap_prompt_stack 0.64). Одинаковые решения linmap_prompt_stack при разной силе проверены отдельно на задачах переноса: насыщение (разрыв оценок растёт с 3–6 до 17 nats, хук срабатывает), не ошибка. Эффект сосредоточен в точке вызова, одного слоя хватает.

5) Цена подготовки (замер без кэша): linmap 108 с (проход базы по 110 промптам) + 0.3 с подгонки, 35k параметров; linmap_prompt 655 с (генерация учителя, batch 1) + 0.3 с, 35k; LoRA 672 с градиентного обучения, 3.2M; linmap_lora 224 с + цена LoRA. По времени на этом Mac linmap_prompt сопоставим с LoRA (время — генерация учителя), но без обратного прохода, оптимизатора и копии весов.

Ограничения: на переносе 37–38 меняющих и 12–13 инвариантных пар на часть, интервалы широкие; всё — разведка, подтверждать на отложенной выборке.""",
    next_step="""1) Объединить цели: учитель-промпт там, где он прав, иначе центроид верных состояний базы — проверить, держит ли одна карта и знакомые, и новые типы. 2) Выбор на val не видит переноса: предложить Павлу отбор с отложенным типом требований внутри train (leave-one-type-out) — это решение по протоколу. 3) Расширить сетку α для linmap_prompt (3.0 на краю). 4) Стек слоёв в текущем виде снять; распределение бюджета по слоям вернуть, если понадобится удешевить саму LoRA. 5) Test не открыт: решение об открытии — после заморозки набора методов.""",
)))

## 2026-10-31 12:22 · прогон `20261031-010203_prompt-teacher-qwen3-1.7b_40692b`

**Гипотеза:** explore · **Ноутбук:** notebooks/09_prompt_teacher.ipynb · **Модель:** Qwen/Qwen3-1.7B (mps, float16) · **Режим решения:** scored
**Стадии:** data, baseline, probe, vectors, sweep, fit, transfer · **git:** `b5eaa7c` (есть незакоммиченные изменения)

**Зачем:** Учитель-промпт (explicit_state) и несколько слоёв с бюджетом по важности (SimplexLoRA) против LoRA.

**Отличия от конфига по умолчанию:** `act.alphas` = [1.0, 1.5, 2.0, 3.0]; `act.ranks` = [8, 16]; `data.n_tasks` = 160; `episode.on_val` = False; `eval.methods` = ['baseline', 'steering', 'lora', 'explicit_state', 'linmap_lora', 'linmap', 'linmap_prompt', 'linmap_stack', 'linmap_prompt_stack']; `model.name` = 'Qwen/Qwen3-1.7B'; `transfer.episode_methods` = ['baseline', 'lora', 'linmap', 'linmap_prompt', 'linmap_stack', 'linmap_prompt_stack']; `transfer.seed` = 303

**⚠ Флаги протокола:**
- незакоммиченные изменения в src/docs/notebooks/configs — прогон не воспроизводится по коммиту

**Результаты (val):**

| method | success | sensitivity | specificity | diff_vs_base | ci | fixed | broken | tokens | calls | p50_ms |
|---|---|---|---|---|---|---|---|---|---|---|
| baseline | 0.621 | 0.526 | 0.400 | — |  | — | — | 831 | 10.000 | 400.600 |
| steering | 0.655 | 0.632 | 0.400 | 0.034 | [+0.000, +0.086] | 2.000 | 0.000 | 831 | 10.000 | 410.200 |
| lora | 0.948 | 1.000 | 0.800 | 0.328 | [+0.190, +0.466] | 19.000 | 0.000 | 831 | 10.000 | 299.000 |
| explicit_state | 0.552 | 0.474 | 0.100 | -0.069 | [-0.207, +0.069] | 6.000 | 10.000 | 1932 | 11.000 | 3443.700 |
| linmap_lora | 0.845 | 0.789 | 0.700 | 0.224 | [+0.086, +0.379] | 18.000 | 5.000 | 831 | 10.000 | 425.800 |
| linmap | 0.862 | 0.842 | 0.600 | 0.241 | [+0.121, +0.379] | 16.000 | 2.000 | 831 | 10.000 | 410.600 |
| linmap_prompt | 0.741 | 0.632 | 0.600 | 0.121 | [+0.017, +0.241] | 9.000 | 2.000 | 831 | 10.000 | 297.400 |
| linmap_stack | 0.793 | 0.842 | 0.500 | 0.172 | [+0.069, +0.293] | 10.000 | 0.000 | 831 | 10.000 | 307.400 |
| linmap_prompt_stack | 0.707 | 0.632 | 0.500 | 0.086 | [+0.000, +0.190] | 6.000 | 1.000 | 831 | 10.000 | 301.400 |

**Перенос, seen (разведка):**

| method | success | sensitivity | specificity | diff_vs_base | ci | fixed | broken | episode | ep_diff | tokens | calls | p50_ms |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| baseline | 0.560 | 0.132 | 0.583 | — |  | — | — | 0.560 | — | 861 | 10.000 | 299.900 |
| steering | 0.550 | 0.132 | 0.583 | -0.010 | [-0.060, +0.040] | 5.000 | 6.000 | — | — | 861 | 10.000 | 302.600 |
| lora | 0.850 | 0.632 | 1.000 | 0.290 | [+0.190, +0.390] | 33.000 | 4.000 | 0.760 | 0.200 | 861 | 10.000 | 309.200 |
| explicit_state | 0.720 | 0.368 | 0.833 | 0.160 | [+0.070, +0.250] | 19.000 | 3.000 | — | — | 2032 | 11.000 | 4371.800 |
| linmap_lora | 0.680 | 0.316 | 0.667 | 0.120 | [+0.030, +0.220] | 26.000 | 14.000 | — | — | 861 | 10.000 | 301.200 |
| linmap | 0.760 | 0.605 | 0.667 | 0.200 | [+0.120, +0.280] | 28.000 | 8.000 | 0.680 | 0.120 | 861 | 10.000 | 309.000 |
| linmap_prompt | 0.700 | 0.289 | 0.833 | 0.140 | [+0.050, +0.230] | 19.000 | 5.000 | 0.630 | 0.070 | 861 | 10.000 | 311.800 |
| linmap_stack | 0.690 | 0.342 | 0.750 | 0.130 | [+0.060, +0.210] | 15.000 | 2.000 | 0.660 | 0.100 | 861 | 10.000 | 323.900 |
| linmap_prompt_stack | 0.660 | 0.289 | 0.750 | 0.100 | [+0.030, +0.180] | 11.000 | 1.000 | 0.640 | 0.080 | 861 | 10.000 | 323.000 |

**Перенос, unseen (разведка):**

| method | success | sensitivity | specificity | diff_vs_base | ci | fixed | broken | episode | ep_diff | tokens | calls | p50_ms |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| baseline | 0.450 | 0.054 | 0.385 | — |  | — | — | 0.330 | — | 840 | 10.000 | 297.500 |
| steering | 0.500 | 0.108 | 0.462 | 0.050 | [-0.030, +0.130] | 9.000 | 4.000 | — | — | 840 | 10.000 | 303.100 |
| lora | 0.880 | 0.757 | 0.846 | 0.430 | [+0.320, +0.540] | 45.000 | 2.000 | 0.850 | 0.520 | 840 | 10.000 | 312.600 |
| explicit_state | 0.910 | 0.838 | 0.846 | 0.460 | [+0.360, +0.560] | 50.000 | 4.000 | — | — | 1988 | 11.000 | 4141.200 |
| linmap_lora | 0.770 | 0.568 | 0.692 | 0.320 | [+0.180, +0.460] | 40.000 | 8.000 | — | — | 840 | 10.000 | 298.800 |
| linmap | 0.640 | 0.243 | 0.692 | 0.190 | [+0.100, +0.280] | 20.000 | 1.000 | 0.460 | 0.130 | 840 | 10.000 | 300.900 |
| linmap_prompt | 0.830 | 0.622 | 0.846 | 0.380 | [+0.270, +0.500] | 41.000 | 3.000 | 0.720 | 0.390 | 840 | 10.000 | 300.800 |
| linmap_stack | 0.610 | 0.162 | 0.692 | 0.160 | [+0.080, +0.250] | 17.000 | 1.000 | 0.410 | 0.080 | 840 | 10.000 | 311.900 |
| linmap_prompt_stack | 0.640 | 0.270 | 0.692 | 0.190 | [+0.110, +0.280] | 19.000 | 0.000 | 0.570 | 0.240 | 840 | 10.000 | 312.200 |

**Вердикты (автоматически, по правилам HYPOTHESES.md):**
- H1: ✔ support — лучший probe L16/post_message: AUROC 0.980, acc 0.882 при перестановке 0.573
- H2: ✔ support — различает (H1), но ошибается в 47% меняющих пар: чувствительность 0.526, специфичность 0.400
- H3: ? inconclusive (val) — на val: успех 0.621 → 0.655 при L16/pre_action/α=1.0; решает только test
- H4: ✘ against (val) — LoRA не хуже steering на условном срезе: steering − LoRA -0.293 [-0.431, -0.155]
- ALT: ? inconclusive (val) — то же качество, что у steering, но дороже: explicit_state (steering − explicit_state +0.103 [-0.034, +0.241]), но дороже: p50 3444 vs 410 мс

**Вывод:** РАЗВЕДКА (hypothesis=explore, test не открыт). Цели для аффинной поправки в точке вызова без LoRA-учителя, Qwen3-1.7B, v3, те же 110 промптов train; перенос — на свежем наборе seed 303 (набор 202 повлиял на выбор методов).

1) Учитель-промпт (linmap_prompt, выбран L20, ранг 8, α=3.0 — край сетки). Сам учитель explicit_state на train не лучше базы (0.736 против 0.755), на val хуже (0.552, специф. 0.1) — поэтому на val linmap_prompt 0.741 уступает linmap 0.862. Но на переносе (успех / чувств. / специф.; эпизоды):
- новые типы требований: база 0.45/0.05/0.38 (эп. 0.33); LoRA 0.88/0.76/0.85 (0.85); explicit_state 0.91/0.84/0.85; linmap_prompt 0.83/0.62/0.85 (0.72, +0.39 [+0.26, +0.52]); linmap_lora 0.77/0.57/0.69; linmap 0.64/0.24/0.69 (0.46).
- знакомые типы: база 0.56/0.13/0.58 (0.56); LoRA 0.85/0.63/1.00 (0.76); explicit_state 0.72/0.37/0.83; linmap 0.76/0.61/0.67 (0.68); linmap_prompt 0.70/0.29/0.83 (0.63, +0.07, интервал включает 0); linmap_lora 0.68/0.32/0.67.
На новых типах поправка от учителя-промпта даёт ~90% одношагового прироста LoRA (+0.38 против +0.43) и ~75% в эпизодах (+0.39 против +0.52) без градиента и без LoRA; при выводе цена как у базы (~300 мс, 861 токен), а сам explicit_state — 4.1 с и 2.4× токенов. Цели дополняют друг друга: центроид верных состояний (linmap) лучше на знакомых типах, учитель-промпт — на новых.

2) Val не предсказал перенос: на val только знакомые типы, выбор по нему не видит переносимости (linmap_prompt на val ниже linmap, на новых типах выше на 0.19).

3) Повтор на свежем наборе: linmap 0.76 / 0.64 (в 08: 0.85 / 0.61), linmap_lora 0.68 / 0.77 (0.73 / 0.78) — картина та же.

4) Несколько слоёв (идея SimplexLoRA): важность слоёв почти ровная (прирост одного слоя на val 0.14–0.24 для linmap, 0.09–0.12 для linmap_prompt), доли 0.12–0.21 против равных 0.167 — оба распределения дают одно и то же, выбрано равномерное. Стек хуже лучшего одного слоя и на val, и на переносе (новые типы: linmap_stack 0.61, linmap_prompt_stack 0.64). Одинаковые решения linmap_prompt_stack при разной силе проверены отдельно на задачах переноса: насыщение (разрыв оценок растёт с 3–6 до 17 nats, хук срабатывает), не ошибка. Эффект сосредоточен в точке вызова, одного слоя хватает.

5) Цена подготовки (замер без кэша): linmap 108 с (проход базы по 110 промптам) + 0.3 с подгонки, 35k параметров; linmap_prompt 655 с (генерация учителя, batch 1) + 0.3 с, 35k; LoRA 672 с градиентного обучения, 3.2M; linmap_lora 224 с + цена LoRA. По времени на этом Mac linmap_prompt сопоставим с LoRA (время — генерация учителя), но без обратного прохода, оптимизатора и копии весов.

Ограничения: на переносе 37–38 меняющих и 12–13 инвариантных пар на часть, интервалы широкие; всё — разведка, подтверждать на отложенной выборке.

**Следующий шаг:** 1) Объединить цели: учитель-промпт там, где он прав, иначе центроид верных состояний базы — проверить, держит ли одна карта и знакомые, и новые типы. 2) Выбор на val не видит переноса: предложить Павлу отбор с отложенным типом требований внутри train (leave-one-type-out) — это решение по протоколу. 3) Расширить сетку α для linmap_prompt (3.0 на краю). 4) Стек слоёв в текущем виде снять; распределение бюджета по слоям вернуть, если понадобится удешевить саму LoRA. 5) Test не открыт: решение об открытии — после заморозки набора методов.
